## Gold: netto_incidence_per_land_2024.

The headline analytical output: per-Bundesland net EEG incidence.

Method:
  flow_a = SUM(verguetung_eur) per bundesland from silver.fct_eeg_zahlung_jaehrlich
  flow_b = total_eeg_burden * (bund_anteil_share per bundesland)
  netto  = flow_a - flow_b

The total EEG financing burden in 2024 was 18,489 Mio EUR (BMF / EEG-Konto, 2024).
This is the amount the Bundeshaushalt (KTF) actually transferred to ÜNB in 2024.

Source: pv-magazine.de, EEG-Konto Jahresabschluss 2024, citing netztransparenz.de.

In [0]:
%sql
DROP TABLE IF EXISTS eeg_dev.gold.netto_incidence_per_land_2024;

In [0]:
from pyspark.sql import functions as F


# Total EEG financing burden 2024 in EUR (= 18,489 Mio EUR per published EEG-Konto data)
TOTAL_EEG_BURDEN_EUR = 18_489_000_000.0


# Flow A: payments received per Bundesland (from silver fct)
flow_a = (
    spark.table("eeg_dev.silver.fct_eeg_zahlung_jaehrlich")
    .filter(F.col("jahr") == 2024)
    .groupBy("bundesland")
    .agg(
        F.sum("verguetung_eur").alias("flow_a_eur"),
        F.countDistinct("payment_key").alias("anlagen_anzahl"),
    )
)

# Flow B: financing burden allocated by Bund tax share
dim = (
    spark.table("eeg_dev.silver.dim_bundesland")
    .withColumn(
        "flow_b_eur",
        F.col("bund_anteil_share") * F.lit(TOTAL_EEG_BURDEN_EUR),
    )
)

# Outer join so offshore (no Flow B) and any zero-payment Land are both visible
gold = (
    flow_a.alias("a")
    .join(dim.alias("d"), F.col("a.bundesland") == F.col("d.bundesland"), how="outer")
    .select(
        F.coalesce(F.col("a.bundesland"), F.col("d.bundesland")).alias("bundesland"),
        F.col("d.einwohner"),
        F.coalesce(F.col("a.flow_a_eur"), F.lit(0.0)).alias("verguetung_erhalten_eur"),
        F.coalesce(F.col("d.flow_b_eur"), F.lit(0.0)).alias("finanzierungslast_eur"),
        F.col("a.anlagen_anzahl"),
    )
    .withColumn(
        "netto_position_eur",
        F.col("verguetung_erhalten_eur") - F.col("finanzierungslast_eur"),
    )
    .withColumn(
        "verguetung_erhalten_eur_pro_kopf",
        F.when(F.col("einwohner").isNotNull(),
               F.col("verguetung_erhalten_eur") / F.col("einwohner"))
    )
    .withColumn(
        "finanzierungslast_eur_pro_kopf",
        F.when(F.col("einwohner").isNotNull(),
               F.col("finanzierungslast_eur") / F.col("einwohner"))
    )
    .withColumn(
        "netto_position_eur_pro_kopf",
        F.when(F.col("einwohner").isNotNull(),
               F.col("netto_position_eur") / F.col("einwohner"))
    )
)

(
    gold.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("eeg_dev.gold.netto_incidence_per_land_2024")
)


# ----- Headline output -----------------------------------------------------

result = (
    spark.table("eeg_dev.gold.netto_incidence_per_land_2024")
    .orderBy(F.desc("netto_position_eur"))
    .collect()
)

print("=" * 95)
print("Netto EEG-Incidence per Bundesland 2024")
print("=" * 95)
print(f"\n{'Bundesland':<28}  {'Erhalten':>12}  {'Last':>12}  {'Netto':>12}  {'Netto/Kopf':>12}")
print(f"{'':<28}  {'(Mio €)':>12}  {'(Mio €)':>12}  {'(Mio €)':>12}  {'(€/Kopf)':>12}")
print("-" * 95)

for r in result:
    erhalten_m = (r["verguetung_erhalten_eur"] or 0) / 1e6
    last_m = (r["finanzierungslast_eur"] or 0) / 1e6
    netto_m = (r["netto_position_eur"] or 0) / 1e6
    pcap = r["netto_position_eur_pro_kopf"]
    pcap_str = f"{pcap:>12,.0f}" if pcap is not None else f"{'-':>12}"
    print(f"{r['bundesland']:<28}  {erhalten_m:>12,.0f}  {last_m:>12,.0f}  {netto_m:>12,.0f}  {pcap_str}")